# generate a C array from sampled data
This notebook extracts data from a csv file, could be data collecterd from your accelerometer.
Tha data is saved as an array in a separate `.h` file as follows

```c
const float samples[3][150] = {
    {
        -0.010000, 0.010000, ...    
    },
    {
        -1.020000, -0.910000, ...
    },
    {
        0.080000, 0.070000, ...
    }
};

```

In [32]:
import numpy as np
import matplotlib.pyplot as plt
import csv

In [33]:
import numpy as np

def generate_c_array_from_csv(filename):
    # Load data from CSV file, assuming the first row is a header
    data = np.genfromtxt(filename, delimiter=",", skip_header=1)

    # Extract the relevant 100 samples (indices 100 to 200)
    accX = data[50:150, 1]
    accY = data[50:150, 2]
    accZ = data[50:150, 3]
    gyroX = data[50:150, 4]
    gyroY = data[50:150, 5]
    gyroZ = data[50:150, 6]

    # Generate the C array code
    c_array_code = f"""

const float samples[6][100] = {{
    {{
        {', '.join(f"{x:.6f}" for x in accX)}
    }},
    {{
        {', '.join(f"{y:.6f}" for y in accY)}
    }},
    {{
        {', '.join(f"{y:.6f}" for y in accZ)}
    }},
    {{
        {', '.join(f"{y:.6f}" for y in gyroX)}
    }},
    {{
        {', '.join(f"{y:.6f}" for y in gyroY)}
    }},
    {{
        {', '.join(f"{z:.6f}" for z in gyroZ)}
    }}
}};

"""

    return c_array_code


In [119]:

# Example usage
# filename = "./tcp_server/out_converted/saved/Break.00105.csv"  # Update this path
# filename = "./tcp_server/out_converted/saved/Left.00137.csv"  # Update this path
# filename = "./tcp_server/out_converted/saved/Right.00121.csv"  # Update this path
filename = "./tcp_server/out_converted/saved/Straight.00093.csv"  # Update this path
c_code = generate_c_array_from_csv(filename)

# Save to a .c file or print
#with open("stairs.h", "w") as file:
# with open("Break.h", "w") as file:
#     file.write(c_code)

print("C code generated and saved to h file")


C code generated and saved to h file


In [120]:
c_code

'\n\nconst float samples[6][100] = {\n    {\n        1.040000, -4.450000, 5.630000, 0.800000, -2.180000, 1.490000, -1.990000, 1.420000, -2.680000, 0.500000, 2.490000, 4.370000, -1.910000, 1.720000, -7.970000, 0.000000, -2.180000, 1.760000, 2.070000, -1.110000, 1.990000, -0.340000, -0.270000, -0.300000, 1.270000, -1.230000, -2.840000, 1.000000, -2.380000, -1.230000, 2.950000, -1.680000, 1.760000, -0.270000, 0.000000, -1.340000, -0.840000, 1.110000, -1.530000, -1.460000, 0.420000, 0.340000, -1.760000, 0.570000, 0.110000, -3.870000, 3.290000, 1.910000, 1.230000, 0.770000, 1.910000, -1.570000, -1.300000, -0.340000, -0.690000, -4.020000, 1.650000, -0.040000, -1.760000, 2.140000, -5.900000, 1.150000, 1.150000, -3.830000, 6.210000, -9.420000, 5.790000, 6.400000, -3.640000, 5.020000, 0.770000, 0.840000, 1.270000, -3.220000, 0.110000, -2.720000, -2.720000, 2.570000, -1.150000, -0.730000, -2.450000, 1.070000, -1.680000, -1.680000, -3.870000, -0.690000, -2.030000, 0.840000, 0.840000, -3.600000, -

In [121]:
# # Save to a .c file or print
# with open("generated_samples.c", "w") as file:
#     file.write(c_code)

# print("C code generated and saved to generated_samples.c")

In [122]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
scaler = MinMaxScaler(feature_range=(-2**15,2**15-1))

columns = ['timestamp', 'accX', 'accY', 'accZ', 'gyroX', 'gyroY', 'gyroZ']
dfs = list()

scaler.fit([[50,50,50,5,5,5],[-50,-50,-50,-5,-5,-5]])

har_df = pd.read_csv(filename, header = 0, names = columns, on_bad_lines='skip')
dfs.append(har_df)

df = pd.concat(dfs, ignore_index=True)

df[['accX', 'accY', 'accZ', 'gyroX', 'gyroY', 'gyroZ']] = scaler.fit_transform(df[['accX', 'accY', 'accZ', 'gyroX', 'gyroY', 'gyroZ']]).astype(np.int16)

In [123]:
xa_list = []
xg_list = []
ya_list = []
yg_list = []
za_list = []
zg_list = []
train_labels = []

window_size = 100
step_size = 50

# creating overlaping windows of size window-size 120
for i in range(0, df.shape[0] - window_size, step_size):
    xs_acc = df['accX'].values[i: i + window_size]
    ys_acc = df['accY'].values[i: i + window_size]
    zs_acc = df['accZ'].values[i: i + window_size]
    xs_gyro = df['gyroX'].values[i: i + window_size]
    ys_gyro = df['gyroY'].values[i: i + window_size]
    zs_gyro = df['gyroZ'].values[i: i + window_size]
    
    # label = df['activity'][i: i + window_size].mode()[0]

    xa_list.append(xs_acc)
    xg_list.append(xs_gyro)
    ya_list.append(ys_acc)
    yg_list.append(ys_gyro)
    za_list.append(zs_acc)
    zg_list.append(zs_gyro)
    # train_labels.append(label)

# Statistical Features on raw x, y and z in time domain
X_train = pd.DataFrame()

# mean
X_train['xa_mean'] = pd.Series(xa_list).apply(lambda x: x.mean())
X_train['ya_mean'] = pd.Series(ya_list).apply(lambda x: x.mean())
X_train['zg_mean'] = pd.Series(zg_list).apply(lambda x: x.mean())

# max
X_train['xg_max'] = pd.Series(xg_list).apply(lambda x: x.max())

# interquartile range
X_train['za_IQR'] = pd.Series(za_list).apply(lambda x: np.percentile(x, 75) - np.percentile(x, 25))

# signal magnitude area
X_train['sma_gyro'] =    pd.Series(xg_list).apply(lambda x: np.sum(abs(x)/100)) + pd.Series(yg_list).apply(lambda x: np.sum(abs(x)/100)) \
                  + pd.Series(zg_list).apply(lambda x: np.sum(abs(x)/100))

In [124]:
print(X_train.astype(np.int16))

   xa_mean  ya_mean  zg_mean  xg_max  za_IQR  sma_gyro
0     4513      585    -1715   13687    5655     13538
1     4439     1452      640   13687    5418     10496
2     3823     1345     5149   24471    6034     16195
